# All Convolutional Net Regularization


In [6]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics

class DenseConvNet(pl.LightningModule):
    def __init__(self, class_count=10, dropout_ratio=0.5):
        super(DenseConvNet, self).__init__()
        # Initial setup of convolutional layers with dropout for regularization
        self.layer1_conv = nn.Conv2d(1, 96, kernel_size=3, padding=1)
        self.layer2_conv = nn.Conv2d(96, 96, kernel_size=3, padding=1)
        self.layer1_dropout = nn.Dropout(dropout_ratio)
        self.layer3_conv = nn.Conv2d(96, 96, kernel_size=3, stride=2, padding=1)  # Downsampling
        self.layer4_conv = nn.Conv2d(96, 192, kernel_size=3, padding=1)
        self.layer5_conv = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.layer2_dropout = nn.Dropout(dropout_ratio)
        self.layer6_conv = nn.Conv2d(192, 192, kernel_size=3, stride=2, padding=1)  # Further downsampling
        self.layer7_conv = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.layer8_conv = nn.Conv2d(192, 192, kernel_size=1)  # Point-wise convolution
        self.layer9_conv = nn.Conv2d(192, class_count, kernel_size=1)  # Classifying with 1x1 convolution

        self.adaptive_pooling = nn.AdaptiveAvgPool2d(1)  # Global average pooling
        self.class_accuracy = torchmetrics.Accuracy(
            num_classes=class_count, average="macro", task="multiclass"
        )

    def forward(self, inputs):
        inputs = F.relu(self.layer1_conv(inputs))
        inputs = F.relu(self.layer2_conv(inputs))
        inputs = self.layer1_dropout(inputs)  # Apply dropout after the second convolutional layer
        inputs = F.relu(self.layer3_conv(inputs))
        inputs = F.relu(self.layer4_conv(inputs))
        inputs = F.relu(self.layer5_conv(inputs))
        inputs = self.layer2_dropout(inputs)  # Apply dropout after the fifth convolutional layer
        inputs = F.relu(self.layer6_conv(inputs))
        inputs = F.relu(self.layer7_conv(inputs))
        inputs = F.relu(self.layer8_conv(inputs))
        inputs = self.layer9_conv(inputs)
        inputs = self.adaptive_pooling(inputs)
        inputs = inputs.view(inputs.size(0), -1)  # Flattening for output
        return inputs

    def training_step(self, batch, batch_idx):
        data, targets = batch
        predictions = self(data)
        loss = F.cross_entropy(predictions, targets)
        self.log("training_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        data, targets = batch
        predictions = self(data)
        validation_loss = F.cross_entropy(predictions, targets)
        self.class_accuracy(predictions, targets)
        self.log("validation_loss", validation_loss, on_epoch=True, prog_bar=True)
        self.log("validation_accuracy", self.class_accuracy, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        data, targets = batch
        predictions = self(data)
        test_loss = F.cross_entropy(predictions, targets)

        self.class_accuracy(predictions, targets)

        self.log("test_accuracy", self.class_accuracy)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)  # Setup optimizer with a learning rate



The Imagenette dataset is a smaller subset of 10 easily classified classes from Imagenet. It is available to download from `torchvision`, as shown in the cell below. There are 3 different sizes of the images available. Feel free to use whichever version you prefer. It might make a difference in the performance of your model.

**Note: After downloading the Imagenette dataset, you will need to set `download=False` in the cell below to avoid errors.**

In [8]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torchvision import transforms
from torchvision.datasets import Imagenette
import torch
from torch.utils.data import DataLoader, random_split

# Enhanced data augmentation and normalization for the training set
augmented_train_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # Rotates images by up to 10 degrees for augmentation
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),  # Applies random crops of various sizes
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale(),  # Converts images to grayscale for consistent channel handling
])

# Standard transformations for testing to ensure consistency
standard_test_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()
])

# Loading training data from Imagenette with transformations applied
imagenette_training_data = Imagenette("../data/imagenette/train/", split="train", size="160px", download=False, transform=augmented_train_transforms)

# Split the training data to create a validation set
training_data_size = int(0.9 * len(imagenette_training_data))
validation_data_size = len(imagenette_training_data) - training_data_size

# Seed for consistent data splitting
consistency_seed = torch.Generator().manual_seed(42)
training_subset, validation_subset = random_split(imagenette_training_data, [training_data_size, validation_data_size], generator=consistency_seed)
validation_subset.dataset.transform = standard_test_transforms

# DataLoader setups for handling training and validation subsets
training_loader = DataLoader(
    training_subset, batch_size=128, num_workers=8, shuffle=True, persistent_workers=True
)
validation_loader = DataLoader(
    validation_subset, batch_size=128, num_workers=8, shuffle=False, persistent_workers=True
)

# Setup for the test dataset using consistent preprocessing
imagenette_test_data = Imagenette("../data/imagenette/test/", split="val", size="160px", download=False, transform=standard_test_transforms)

# Model initialization with a convolutional neural network
neural_network_model = AllConvNet()

# Monitoring validation loss for early stopping to prevent overfitting
robust_early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5
)

# Configuration for saving the best model based on validation performance
reliable_checkpoint_saving = ModelCheckpoint(
    monitor="val_loss",
    mode="min"
)


In [ ]:
from pathlib import Path


# Fit the model
torch.set_float32_matmul_precision("high")
trainer = L.Trainer(callbacks=[early_stop_callback, checkpoint_callback], max_epochs=-1)

trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# Save the model to be used on the Transfer Learning Model
dir_path = Path("../models")
dir_path.mkdir(parents=True, exist_ok=True)
model_path = dir_path / "all_convnet.pth"
torch.save(model.state_dict(), model_path)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

   | Name            | Type               | Params | Mode 
----------------------------------------------------------------
0  | conv1           | Conv2d             | 960    | train
1  | conv2           | Conv2d             | 83.0 K | train
2  | dropout1        | Dropout            | 0      | train
3  | conv3           | Conv2d             | 83.0 K | train
4  | conv4           | Conv2d             | 166 K  | train
5  | conv5           | Conv2d             | 331 K  | train
6  | dropout2        | Dropout            | 0      | train
7  | conv6           | Conv2d             | 331 K  | train
8  | conv7           | Conv2d             | 331 K  | train
9  | conv8           | Conv2d             | 37.1 K | train
10 | conv9           | Conv2d             | 1.9 K  | train
11 | global_avg_pool | AdaptiveAvgPool2d  | 0      | train
12 | accuracy        | MulticlassAccuracy | 0      |

Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


In [ ]:
# Evaluate the model on the test set
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, num_workers=8, shuffle=False, persistent_workers=True
)
trainer.test(model=model, dataloaders=test_loader)